# Part 4: Statistical Analysis and Hypothesis Testing


## 1. Introduction: From Observations to Facts

Following the exploration of key trends and interesting patterns in the music data in the previous section, the focus now shifts from simple description to concrete proof. The objective is to apply statistical methods to determine whether the observed findings represent consistent patterns or mere coincidences.

The primary objectives of this stage are:
  - To demonstrate whether the discovered relationships in the data are statistically significant.
  - To verify whether the differences between individual genres are mathematically justified.
  - To accurately measure the strength of the correlations between various physical characteristics of sound.

Loading data and libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

# Load and basic clean (same as Part 1 to ensure consistency)
# read_csv('path/to/dataset.csv')
df = pd.read_csv('/content/dataset.csv')
df = df.dropna(subset=['track_name', 'artists'])
df = df[df['tempo'] > 0]
df = df.drop_duplicates(subset=['track_id'])

print("Data re-loaded and ready for statistical testing.")

## 2. Hypothesis Testing 1: Energy vs. Loudness

The exploratory data analysis (EDA) indicated a strong correlation (0.76) between energy and loudness. This phase tests whether the relationship is statistically significant.

  - H0 (Null Hypothesis): There is no correlation between loudness and energy in the song population.
  - H1 (Alternative Hypothesis): There is a significant positive correlation between loudness and energy.

Test Method: Pearson Correlation Test.

In [ ]:
# calculate Pearson correlation coefficient and p-value
r_stat, p_value = stats.pearsonr(df['loudness'], df['energy'])

print(f"Pearson Correlation Coefficient: {r_stat:.4f}")
print(f"P-value: {p_value}")

### Interpretation of Hypothesis 1:
- **Result:** The correlation coefficient is approximately **0.76**, and the **p-value is close to zero** (p < 0.05).
- **Decision:** **We reject the null hypothesis (H0)**.
- **Scientific conclusion:** There is a statistically significant, strong positive correlation between sound intensity and energy.

**Connection to theory:**
This confirms that the physical amplitude of the sound wave (sound intensity) is a key mathematical factor in how “energetic” a piece of music sounds to the listener. The p-value, close to zero, indicates that this is not a random coincidence.

In [ ]:
# visualizing the relationship with a regression line
plt.figure(figsize=(10, 6))
# .sample(2000) is used to prevent the plot from being overcrowded
# and to ensure the red line remains clearly visible
sns.regplot(x='loudness', y='energy', data=df.sample(2000),
            scatter_kws={'alpha':0.2}, line_kws={'color':'red'})
plt.title("Loudness vs Energy (with Regression Line)")
plt.xlabel("Loudness (dB)")
plt.ylabel("Energy")
plt.show()

The line shows the general trend in the data.
As loudness increases, energy also increases.
This confirms the statistical result — the relationship is not random, but follows a clear pattern.

# 3. Hypothesis Testing 2: Genre and Energy Levels

Observations show that different genres possess varying energy levels. This test determines whether these differences are statistically significant across multiple categories.

  - H0 (Null Hypothesis): The average energy level is the same across all musical genres.
  - H1 (Alternative Hypothesis): At least one genre has a significantly different average energy level than the others.

Test Method: One-Way ANOVA (Analysis of Variance).

In [ ]:
# create a bar plot to compare average energy across selected genres
plt.figure(figsize=(10, 6))

# filter the dataset to include only the selected genres
filtered_df = df[df['track_genre'].isin(test_genres)]

# plot mean energy for each genre with confidence intervals
sns.barplot(
    x='track_genre',
    y='energy',
    data=filtered_df,
    hue='track_genre',     # fix for FutureWarning - assigns color by genre
    legend=False,          # hides the legend (not needed here)
    capsize=0.1,           # adds small caps to error bars
    palette='viridis'      # color palette for better visualization
)

# add title and labels
plt.title("Average Energy by Genre with 95% Confidence Intervals")
plt.xlabel("Genre")
plt.ylabel("Mean Energy")

# display the plot
plt.show()

In [ ]:
# print ANOVA test results in a readable format
print("=" * 45)
print("        ANOVA Test Results")
print("=" * 45)
print(f"  F-statistic : {f_stat:.4f}")
print(f"  P-value     : {p_value_anova:.2e}")
print("=" * 45)

# Interpret the result automatically
if p_value_anova < 0.05:
    print("  Result: Reject H0")
    print("  The differences between genres ARE significant.")
else:
    print("  Result: Fail to reject H0")
    print("  The differences between genres are NOT significant.")
print("=" * 45)

### Statistical Analysis and Visual Interpretation of Hypothesis 2

  - **What the Mathematics Tells Us** (Numerical Results)

    - **High level of certainty:** The ANOVA test yielded an extremely low p-value (significantly below the 0.05 threshold) and a high F-statistic.
    - **Conclusion:** The null hypothesis (H0) is rejected.
    - **Key finding:** There is a real, proven difference in energy levels between the selected music genres. This confirms that a song’s genre is mathematically linked to the amount of energy it conveys.

  - **What the graph shows (visual evidence)**

    - **Average energy:** Each bar shows the average energy level. It is easy to see that genres such as alt-rock and Afrobeats are in a much higher “energy zone” than acoustic music or tango.
    - **The black lines (errors):** These lines at the top of the bars represent 95% confidence intervals. They show the range within which the true mean value almost certainly lies.
    - **No overlap, no coincidence:** The error bars for acoustic songs do not touch or overlap with those of alt-rock. This is visual proof that their energy structures are fundamentally different, and this difference is not merely a lucky guess or a random coincidence.

## 4. Hypothesis Testing 3: Tempo vs. Danceability

Does a faster tempo (BPM) make a song more danceable? This test examines the relationship between the speed of the music and its physical rhythm structure.

  - H0 (Null Hypothesis): There is no correlation between tempo and danceability.
  - H1 (Alternative Hypothesis): There is a significant correlation between tempo and danceability.

Test Method: Spearman Rank Correlation.

Before applying the test, the data is first inspected visually to determine whether the relationship between tempo and danceability follows a linear or non-linear pattern. This step justifies the choice of Spearman over Pearson correlation.

In [ ]:
# visual inspection with non-linear trend
plt.figure(figsize=(10, 6))

sample_df = df.sample(2000, random_state=42)

sns.scatterplot(
    data=sample_df,
    x='tempo',
    y='danceability',
    alpha=0.25
)

# LOWESS curve (non-linear trend)
sns.regplot(
    data=sample_df,
    x='tempo',
    y='danceability',
    scatter=False,
    lowess=True,
    color='red'
)

plt.title("Tempo vs Danceability with Non-Linear Trend")
plt.xlabel("Tempo (BPM)")
plt.ylabel("Danceability")
plt.show()

In [ ]:
from scipy import stats

pearson_r, pearson_p = stats.pearsonr(df['tempo'], df['danceability'])
spearman_rho, spearman_p = stats.spearmanr(df['tempo'], df['danceability'])

print("=" * 45)
print("   Correlation Comparison")
print("=" * 45)
print(f"Pearson r        : {pearson_r:.4f}")
print(f"Pearson p-value  : {pearson_p:.4e}")
print(f"Spearman rho     : {spearman_rho:.4f}")
print(f"Spearman p-value : {spearman_p:.4e}")
print("=" * 45)

In [ ]:
# Final Spearman test
rho, p_value_spearman = stats.spearmanr(df['tempo'], df['danceability'])

print(f"Spearman Correlation (rho): {rho:.4f}")
print(f"P-value: {p_value_spearman:.4e}")

### Interpretation of Hypothesis 3:

- **What the numbers say:** The Spearman correlation coefficient is very close to zero, and although the p-value may indicate statistical significance, the strength of the relationship is negligible.
- **Decision:** Even if we reject H0 statistically, the effect size is extremely small.
- **Key Insight:** There is **no meaningful relationship** between tempo and danceability.

**Visual Evidence:**
The scatter plot shows a wide dispersion of points with no clear linear pattern. The red LOWESS curve is not a straight line and does not show a strong increasing or decreasing trend.

**Comparison of Methods:**
The Pearson and Spearman coefficients are both close to zero, confirming that neither a linear nor a monotonic relationship exists.

**Scientific Conclusion:**
This result demonstrates that **speed (tempo) does not determine rhythmic usability (danceability)**. Music is a multi-dimensional system, where movement depends on complex rhythmic structures rather than tempo alone.

# 5. General Project Conclusion


This project explores how music theory, the physics of sound, and mathematics come together using data from over 114,000 pieces of music.

## 5.1 Project Summary

Music as Physics: The results confirm that music is more than just an art form—it is a structured physical phenomenon built on resonance, frequencies, and mathematical relationships.
The Energy-Volume Relationship: Data analysis demonstrates that what is perceived as “energy” in a song is directly related to its physical volume. The high correlation (0.76) indicates that this is a consistent rule, not a coincidence.
Genres as Mathematical Profiles: Statistical tests (ANOVA) show that genres are not simply cultural labels. They are distinct mathematical profiles with their own unique sonic characteristics.
The Complexity of Rhythm: Analysis of tempo and danceability reveals that music is a multidimensional system where the desire to move depends on complex rhythmic structures, not just speed (BPM).

## 5.2 Scientific Value

Going from theoretical concepts to evidence - an approach built on three pillars:

Discovery: Transforming musical elements into recognizable mathematical patterns.

Validation: Supporting each insight with a vast set of real-world data.

Precision: Using professional statistical methods to ensure that all findings are mathematically sound and valid.

## 5.3 Final Thoughts

The connection between mathematics and music is deep. While physics explains how a string vibrates, statistics helps to understand how millions of songs across cultures still follow predictable patterns.

Where words fail, music speaks – and where music speaks, mathematics proves.